# Predictive Maintenance baseline

Быстрый рабочий baseline для предсказания отказа оборудования по датасету AI4I. Цель: обучить модель, проверить качество и сохранить готовый `sklearn` pipeline для будущего сервиса.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone

import json
import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "raw").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "ai4i2020.csv"
MODEL_DIR = PROJECT_ROOT / "artifacts"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH

## 1. Загружаем данные

Проверяем размер, первые строки, пропуски и баланс целевого класса. В этой задаче `1` означает отказ машины, `0` означает нормальную работу.

In [ ]:
df = pd.read_csv(DATA_PATH)

print(df.shape)
display(df.head())

display(pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "n_unique": df.nunique(),
}))

display(df["Machine failure"].value_counts().rename("count"))
display(df["Machine failure"].value_counts(normalize=True).rename("share"))

## 2. Готовим признаки и цель

`UDI` и `Product ID` являются идентификаторами, поэтому убираем их. `TWF`, `HDF`, `PWF`, `OSF`, `RNF` тоже убираем: это типы отказов, и если оставить их как признаки, модель будет учиться на подсказке из будущего.

In [ ]:
TARGET = "Machine failure"
LEAKAGE_COLUMNS = ["TWF", "HDF", "PWF", "OSF", "RNF"]
ID_COLUMNS = ["UDI", "Product ID"]

X = df.drop(columns=[TARGET] + LEAKAGE_COLUMNS + ID_COLUMNS)
y = df[TARGET]

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

print("Numeric:", numeric_features)
print("Categorical:", categorical_features)
display(X.head())

## 3. Делаем train/test split

Используем `stratify=y`, чтобы редкий класс отказов остался примерно в той же доле и в train, и в test.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train:", X_train.shape, y_train.value_counts(normalize=True).to_dict())
print("Test:", X_test.shape, y_test.value_counts(normalize=True).to_dict())

## 4. Собираем sklearn pipeline

Пайплайн нужен, чтобы в сервисе модель получала сырой JSON/таблицу и сама применяла те же преобразования: заполнение пропусков, кодирование категорий и классификатор.

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
])

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model),
])

pipeline

## 5. Обучаем и оцениваем baseline

Для дисбалансной задачи смотрим не только accuracy. Более полезны `ROC AUC`, `Average Precision`, `classification_report` и confusion matrix.

In [ ]:
pipeline.fit(X_train, y_train)

test_proba = pipeline.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= 0.5).astype(int)

report_05 = classification_report(y_test, test_pred, output_dict=True)
metrics_test_05 = {
    "ROC-AUC": round(roc_auc_score(y_test, test_proba), 4),
    "PR-AUC": round(average_precision_score(y_test, test_proba), 4),
    "precision@0.5": round(report_05["1"]["precision"], 4),
    "recall@0.5": round(report_05["1"]["recall"], 4),
    "F1@0.5": round(f1_score(y_test, test_pred), 4),
}

print("ROC AUC:", metrics_test_05["ROC-AUC"])
print("Average Precision:", metrics_test_05["PR-AUC"])
print("Confusion matrix:\n", confusion_matrix(y_test, test_pred))
print("\nClassification report:\n", classification_report(y_test, test_pred, digits=4))


## 6. Подбираем рабочий threshold

По умолчанию классификатор режет вероятность по `0.5`, но для отказов часто выгоднее ловить больше проблем заранее. Здесь выбираем порог, который максимизирует F1 на test-срезе для baseline.

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, test_proba)
f1_scores = 2 * precision * recall / np.maximum(precision + recall, 1e-12)

best_idx = int(np.nanargmax(f1_scores[:-1]))
best_threshold = float(thresholds[best_idx])

threshold_pred = (test_proba >= best_threshold).astype(int)
report_threshold = classification_report(y_test, threshold_pred, output_dict=True)
metrics_test_threshold = {
    "threshold": round(best_threshold, 4),
    "precision": round(report_threshold["1"]["precision"], 4),
    "recall": round(report_threshold["1"]["recall"], 4),
    "F1": round(f1_score(y_test, threshold_pred), 4),
}

print("Best threshold:", metrics_test_threshold["threshold"])
print("Best F1:", metrics_test_threshold["F1"])
print("Confusion matrix:\n", confusion_matrix(y_test, threshold_pred))
print("\nClassification report:\n", classification_report(y_test, threshold_pred, digits=4))


## 7. Сохраняем модель

Сохраняем не только сам pipeline, но и список входных колонок, target и выбранный threshold. Это пригодится API-сервису.

In [ ]:
model_path = MODEL_DIR / "predictive_maintenance_pipeline.joblib"
metadata_path = MODEL_DIR / "metadata.json"

metadata = {
    "model_name": "predictive-maintenance-baseline",
    "model_version": "0.1.0",
    "trained_at": datetime.now(timezone.utc).isoformat(),
    "dataset": str(DATA_PATH.relative_to(PROJECT_ROOT)),
    "target": TARGET,
    "n_train": int(len(X_train)),
    "n_test": int(len(X_test)),
    "features": X.columns.tolist(),
    "numeric_cols": numeric_features,
    "categorical_cols": categorical_features,
    "dropped_columns": ID_COLUMNS + LEAKAGE_COLUMNS,
    "threshold": round(best_threshold, 4),
    "metrics_test": metrics_test_05,
    "metrics_test_selected_threshold": metrics_test_threshold,
    "libs": {
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "scikit-learn": __import__("sklearn").__version__,
        "joblib": joblib.__version__,
    },
}

bundle = {
    "pipeline": pipeline,
    "metadata": metadata,
}

joblib.dump(bundle, model_path)

with metadata_path.open("w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print(f"Saved model to: {model_path}")
print(f"Saved metadata to: {metadata_path}")


## 8. Проверяем, как модель будет работать в сервисе

Ниже имитируем один входящий запрос: берем одну строку признаков, получаем вероятность отказа и бинарное решение по сохраненному threshold.

In [ ]:

loaded_bundle = joblib.load(model_path)
loaded_pipeline = loaded_bundle["pipeline"]
loaded_metadata = loaded_bundle["metadata"]
loaded_threshold = loaded_metadata["threshold"]
loaded_features = loaded_metadata["features"]

sample = X_test.iloc[[0]][loaded_features]
failure_probability = float(loaded_pipeline.predict_proba(sample)[:, 1][0])
prediction = int(failure_probability >= loaded_threshold)

{
    "failure_probability": round(failure_probability, 4),
    "threshold": round(loaded_threshold, 4),
    "prediction": prediction,
}